# Cell Neighborhood Tokenization

- Creator: Sebastian Birk (sebastian.birk@helmholtz-munich.de).
- Date of Creation: 23.04.2024
- Date of Last Modification: 04.06.2024

## 1. Setup

### 1.1 Import Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import os
import pickle
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scgpt as scg
import seaborn as sns
import squidpy as sq
import torch
from datasets import load_from_disk
from scipy.stats import spearmanr
from scgpt.model import TransformerModel
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from skmisc.loess import loess

from nichejepa import tokenizers
from nichejepa import utils

/home/aih/sebastian.birk/miniconda3/envs/nichejepa/lib/python3.10/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/home/aih/sebastian.birk/miniconda3/envs/nichejepa/lib/python3.10/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


### 1.2 Run Notebook Setup

In [3]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### 1.3 Configure Paths and Create Directories

In [4]:
# Define paths
st_data_folder_path = "../datasets/st_data" # spatially resolved transcriptomics data
st_data_bronze_folder_path = f"{st_data_folder_path}/bronze"
st_data_silver_folder_path = f"{st_data_folder_path}/silver"
st_data_gold_folder_path = f"{st_data_folder_path}/gold"
tokenizer = "cell_neighborhood_tokenizer"
tokenizer_folder_path = f"../datasets/tokenizer/{tokenizer}"
results_folder_path = "../datasets/results"
geneformer_token_dictionary_file = f"../datasets/tokenizer/geneformer_token_dictionary.pkl"
# dataset_cache_file = "../datasets/scgpt_gene_embeddings"
scgpt_model_dir = Path("../scgpt/pretrained_model")
scgpt_model_config_file = scgpt_model_dir / "args.json"
scgpt_model_file = scgpt_model_dir / "best_model.pt"
scgpt_vocab_file = scgpt_model_dir / "vocab.json"
scgpt_metadata_file = Path("../scgpt/metadata/gene_info.csv")

## 2. Tokenization

In [5]:
# Define dataset
dataset_name = "nanostring_cosmx_human_brain"
#dataset_name = "xenium_human_brain_cancer"

In [6]:
os.makedirs(f"{tokenizer_folder_path}/{dataset_name}", exist_ok=True)

# Set dataset-specific paths and parameters
cell_gene_means_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/cell_gene_means_dictionary.pkl"
cell_gene_nzmedians_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/cell_gene_nzmedians_dictionary.pkl"
cell_gene_logmeans_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/cell_gene_logmeans_dictionary.pkl"
neighborhood_gene_means_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/neighborhood_gene_means_dictionary.pkl"
neighborhood_gene_nzmedians_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/neighborhood_gene_nzmedians_dictionary.pkl"
neighborhood_gene_logmeans_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/neighborhood_gene_logmeans_dictionary.pkl"
nichejepa_token_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/token_dictionary.pkl"

input_directory = f"{st_data_silver_folder_path}/{dataset_name}"
output_directory = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}"
if dataset_name == "nanostring_cosmx_human_brain":
    custom_attr_name_dict = {"cell_type": "cell_types",
                             "niche_type": "niche_types"}
    label_cats = ["cell_types", "niche_types"]
elif dataset_name == "xenium_human_brain_cancer":
    custom_attr_name_dict = {"cluster": "cluster"}
    label_cats = ["cluster"]

### 2.1 Create Token Dictionary

In [7]:
# Load data
adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")

In [8]:
### Create token dict for nichejepa with cell and neighborhood tokens ###

# Load geneformer token dict to get protein-coding and miRNA genes
with open(geneformer_token_dictionary_file, "rb") as f:
    token_dict = pickle.load(f)

# Add cls token
def insert_at_index(d, index, key, value):
    items = list(d.items())
    items.insert(index, (key, value))
    new_d = dict(items)
    return new_d
token_dict = insert_at_index(token_dict, 2, "<cls_cell>", 2)
token_dict = insert_at_index(token_dict, 3, "<cls_neighborhood>", 3)

# Remove genes not in dataset
gene_list = adata.var["ensembl_id"].tolist()
token_dict_copy = token_dict.copy()
for token in token_dict_copy.keys():
    if "ENS" in token and token not in gene_list:
        del(token_dict[token])
del(token_dict_copy)

# Separate gene tokens into cell and neighborhood tokens
new_token_dict = {
    (key + "_cell" if "ENS" in key else key): value
    for value, key in enumerate(token_dict.keys())}
new_token_dict_copy = new_token_dict.copy()
i = 0
for key in new_token_dict_copy.keys():
    if "ENS" in key:
        new_token_dict[key.replace("cell", "neighborhood")] = len(new_token_dict_copy) + i
        i += 1
del(new_token_dict_copy)

# Write to disk
with open(nichejepa_token_dictionary_file, "wb") as file:
    pickle.dump(new_token_dict, file)

In [9]:
# Load vocab
with open(nichejepa_token_dictionary_file, "rb") as file:
    token_dict = pickle.load(file)

token_dict_reverse = {value: key for key, value in token_dict.items()}

In [10]:
# NicheJEPA token dict
len(token_dict)

12066

In [11]:
# First cell gene token
token_dict["ENSG00000000938_cell"]

4

In [12]:
# First neighborhood gene token
token_dict["ENSG00000000938_neighborhood"]

6035

### 2.2 Compute Normalization Factors for Gene Counts

Corpus means, non-zero medians, or log means are used depending on the 'norm_method' specified in the CellNeighborhoodRankTokenizer (check source code).

In [13]:
# Compute Corpus Means, Regularized Standard Deviations, and Non-zero Medians for
# Cell and Neighborhood Genes
for norm_factor in ["read_depth", "cell_area"]:
    # Load data
    adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")

    # Filter cells that did not pass QC
    if "filter_pass" in adata.obs.columns:
        filter_pass_idx = np.where([filter_pass == 1 for filter_pass in adata.obs["filter_pass"]])[0]
    else:
        print(f"adata has no column 'filter_pass'; utilizing all cells.")
        filter_pass_idx = np.array([i for i in range(adata.shape[0])])
    adata = adata[filter_pass_idx]
    
    # Compute spatial neighborhood graph based on Visium spot diameter of 55 microns
    sq.gr.spatial_neighbors(adata,
                            coord_type="generic",
                            spatial_key="spatial",
                            radius=27.5,
                            )
    
    # Add self loops to spatial neighborhood graph
    adata.obsp["spatial_connectivities"].setdiag(1)
        
    # Compute sum of counts across each cell's neighborhood to get neighborhood counts per cell
    adata.layers["X_neighborhood"] = adata.obsp["spatial_connectivities"].T @ adata.X

    if norm_factor == "read_depth":
        # Normalize cell gene counts by read depth
        adata.X = adata.X / adata.X.sum(1).reshape(-1, 1) * 10000
    
        # Normalize neighborhood gene counts by read depth
        adata.layers["X_neighborhood"] = (
            adata.layers["X_neighborhood"] /
            adata.layers["X_neighborhood"].sum(1).reshape(-1, 1) * 10000)
    elif norm_factor == "cell_area":
        # Normalize cell gene counts by cell area
        adata.X = adata.X / adata.obs["cell_area"].values.reshape(-1, 1)
    
        # Compute neighborhood area
        adata.obs["neighborhood_cell_area"] = np.array(adata.obsp["spatial_connectivities"].T @ adata.obs["cell_area"].values.reshape(-1, 1))  
    
        # Normalize neighborhood counts
        adata.layers["X_neighborhood"] = (
            adata.layers["X_neighborhood"] /
            adata.obs["neighborhood_cell_area"].values.reshape(-1, 1))
        
    # Compute means, vars, and non-zero medians of
    # read-depth or cell-area normalized cell gene counts
    cell_gene_means = np.mean(adata.X, axis=0)
    masked_data = np.ma.masked_equal(adata.X, 0)
    cell_gene_nzmedians = np.ma.mean(masked_data, axis=0)
    cell_gene_nzmedians = cell_gene_nzmedians.filled(0)
    
    # Compute means, vars, and non-zero medians of
    # read-depth or cell-area normalized neighborhood gene counts
    neighborhood_gene_means = np.mean(adata.layers["X_neighborhood"], axis=0)
    masked_data_neighborhood = np.ma.masked_equal(adata.layers["X_neighborhood"], 0)
    neighborhood_gene_nzmedians = np.ma.mean(masked_data_neighborhood, axis=0)
    neighborhood_gene_nzmedians = neighborhood_gene_nzmedians.filled(0)

    sc.pp.log1p(adata)
    sc.pp.log1p(adata, layer="X_neighborhood")
    cell_gene_logmeans = np.mean(adata.X, axis=0)
    neighborhood_gene_logmeans = np.mean(adata.layers["X_neighborhood"], axis=0)

    cell_gene_logmeans = np.array(cell_gene_logmeans).reshape(-1)
    cell_gene_nzmedians = np.array(cell_gene_nzmedians).reshape(-1)
    cell_gene_means = np.array(cell_gene_means).reshape(-1)
    neighborhood_gene_means = np.array(neighborhood_gene_means).reshape(-1)
    neighborhood_gene_nzmedians = np.array(neighborhood_gene_nzmedians).reshape(-1)
    neighborhood_gene_logmeans = np.array(neighborhood_gene_logmeans).reshape(-1)
    
    # Create norm factor dictionaries for tokenizer and store on disk
    cell_gene_means_dict = {}
    cell_gene_nzmedians_dict = {}
    cell_gene_logmeans_dict = {}
    neighborhood_gene_means_dict = {}
    neighborhood_gene_nzmedians_dict = {}
    neighborhood_gene_logmeans_dict = {}
    
    for (gene,
         cell_gene_mean,
         cell_gene_nzmedian,
         cell_gene_logmean,
         neighborhood_gene_mean,
         neighborhood_gene_nzmedian,
         neighborhood_gene_logmean) in zip(
            adata.var["ensembl_id"].values,
            cell_gene_means,
            cell_gene_nzmedians,
            cell_gene_logmeans,
            neighborhood_gene_means,
            neighborhood_gene_nzmedians,
            neighborhood_gene_logmeans):
        cell_gene_means_dict[gene] = cell_gene_mean
        cell_gene_nzmedians_dict[gene] = cell_gene_nzmedian
        cell_gene_logmeans_dict[gene] = cell_gene_logmean
        neighborhood_gene_means_dict[gene] = neighborhood_gene_mean
        neighborhood_gene_nzmedians_dict[gene] = neighborhood_gene_nzmedian
        neighborhood_gene_logmeans_dict[gene] = neighborhood_gene_logmean
    
    with open(cell_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(cell_gene_means_dict, file)
    with open(cell_gene_nzmedians_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(cell_gene_nzmedians_dict, file)
    with open(cell_gene_logmeans_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(cell_gene_logmeans_dict, file)
    with open(neighborhood_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(neighborhood_gene_means_dict, file)
    with open(neighborhood_gene_nzmedians_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(neighborhood_gene_nzmedians_dict, file)
    with open(neighborhood_gene_logmeans_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(neighborhood_gene_logmeans_dict, file)

adata has no column 'filter_pass'; utilizing all cells.
adata has no column 'filter_pass'; utilizing all cells.


In [14]:
values = []
for key, value in cell_gene_means_dict.items():
    values.append(value)

np.mean(values)

0.0009804076142728077

In [15]:
values = []
for key, value in neighborhood_gene_means_dict.items():
    values.append(value)

np.mean(values)

0.0010260266725189036

### 2.3 Run Tokenizer

In [7]:
# Tokenize data with different normalization methods
for model_input_size in [#16,
                         #32,
                         #64,
                         2048
                        ]:
    for norm_method in ["analytic_pearson_residuals",
                        "mean",
                        "nzmedian",
                        "seurat_v3",
                        "shifted_logmean",
                        "shifted_log",
    ]:
        if norm_method == "analytic_pearson_residuals":
            output_file_prefix = f"{dataset_name}_{norm_method}_{model_input_size}"
    
            tk = tokenizers.CellNeighborhoodRankTokenizer(
                    model_input_size=model_input_size,
                    norm_method=norm_method,
                    cell_gene_means_file=cell_gene_means_dictionary_file.replace("gene", f"gene_read_depth"),
                    cell_gene_nzmedians_file=cell_gene_nzmedians_dictionary_file.replace("gene", f"gene_read_depth"),
                    cell_gene_logmeans_file=cell_gene_logmeans_dictionary_file.replace("gene", f"gene_read_depth"),
                    neighborhood_gene_means_file=neighborhood_gene_means_dictionary_file.replace("gene", f"gene_read_depth"),
                    neighborhood_gene_nzmedians_file=neighborhood_gene_nzmedians_dictionary_file.replace("gene", f"gene_read_depth"),
                    neighborhood_gene_logmeans_file=neighborhood_gene_logmeans_dictionary_file.replace("gene", f"ggene_read_depth"),
                    token_dictionary_file=nichejepa_token_dictionary_file,
                    custom_attr_name_dict=custom_attr_name_dict)
    
            tk.tokenize_data(input_directory=input_directory,
                             output_directory=output_directory,
                             output_file_prefix=output_file_prefix)
            
        elif norm_method != "analytic_pearson_residuals":
            for norm_factor in ["read_depth",
                                "cell_area"
                               ]:
                output_file_prefix = f"{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}"
    
                tk = tokenizers.CellNeighborhoodRankTokenizer(
                        model_input_size=model_input_size,
                        norm_method=norm_method,
                        cell_gene_means_file=cell_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        cell_gene_nzmedians_file=cell_gene_nzmedians_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        cell_gene_logmeans_file=cell_gene_logmeans_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        neighborhood_gene_means_file=neighborhood_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        neighborhood_gene_nzmedians_file=neighborhood_gene_nzmedians_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        neighborhood_gene_logmeans_file=neighborhood_gene_logmeans_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        token_dictionary_file=nichejepa_token_dictionary_file,
                        custom_attr_name_dict=custom_attr_name_dict)
                    
                tk.tokenize_data(input_directory=input_directory,
                                 output_directory=output_directory,
                                 output_file_prefix=output_file_prefix)

Tokenizing '../datasets/st_data/silver/nanostring_cosmx_human_brain/nanostring_cosmx_human_brain.h5ad'...
Filtering cells.
No 'filter_pass' column in 'adata.obs'; returning full adata.
Computing spatial neighborhood graph and aggregating counts.
Normalizing gene expression counts.
Creating Hugging Face dataset...


Map:   0%|          | 0/188686 [00:00<?, ? examples/s]

Saving the dataset (0/10 shards):   0%|          | 0/188686 [00:00<?, ? examples/s]

Tokenizing '../datasets/st_data/silver/nanostring_cosmx_human_brain/nanostring_cosmx_human_brain.h5ad'...
Filtering cells.
No 'filter_pass' column in 'adata.obs'; returning full adata.
Computing spatial neighborhood graph and aggregating counts.
Normalizing gene expression counts.
Creating Hugging Face dataset...


Map:   0%|          | 0/188686 [00:00<?, ? examples/s]

Saving the dataset (0/10 shards):   0%|          | 0/188686 [00:00<?, ? examples/s]

... (23 outputs left out)

Saving the dataset (0/10 shards):   0%|          | 0/188686 [00:00<?, ? examples/s]

Tokenizing '../datasets/st_data/silver/nanostring_cosmx_human_brain/nanostring_cosmx_human_brain.h5ad'...
Filtering cells.
No 'filter_pass' column in 'adata.obs'; returning full adata.
Computing spatial neighborhood graph and aggregating counts.
Normalizing gene expression counts.
Creating Hugging Face dataset...


Map:   0%|          | 0/188686 [00:00<?, ? examples/s]

Saving the dataset (0/10 shards):   0%|          | 0/188686 [00:00<?, ? examples/s]

### 2.4 Test Tokenizer

In [ ]:
norm_factor = "read_depth"
norm_method = "shifted_log" # "seurat_v3", "shifted_log"
model_input_size = 2048
tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}.dataset"
dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)

In [ ]:
dataset

In [ ]:
adata = ad.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")

# Filter cells that did not pass QC
if "filter_pass" in adata.obs.columns:
    filter_pass_idx = np.where([filter_pass == 1 for filter_pass in adata.obs["filter_pass"]])[0]
else:
    filter_pass_idx = np.array([i for i in range(adata.shape[0])])
adata = adata[filter_pass_idx]

# Compute spatial neighborhood graph based on Visium spot diameter of 55 microns
sq.gr.spatial_neighbors(adata,
                        coord_type="generic",
                        spatial_key="spatial",
                        radius=27.5,
                        )

# Add self loops to spatial neighborhood graph
adata.obsp["spatial_connectivities"].setdiag(1)
    
# Compute sum of counts across each cell's neighborhood to get neighborhood counts per cell
adata.layers["X_neighborhood"] = adata.obsp["spatial_connectivities"].T @ adata.X

In [ ]:
if norm_factor == "read_depth":
    # Normalize cell counts
    target_sum = 10_000
    adata.X = adata.X / adata.X.sum(1).reshape(-1, 1) * target_sum

    # Normalize neighborhood counts
    adata.layers["X_neighborhood"] = (
        adata.layers["X_neighborhood"] /
        adata.layers["X_neighborhood"].sum(1).reshape(-1, 1) * target_sum)

In [ ]:
if norm_method == "seurat_v3":
    # Load cell and neighborhood gene corpus means and vars
    with open(cell_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"), "rb") as f:
        cell_gene_means_dict = pickle.load(f)
    with open(cell_gene_reg_stds_dictionary_file.replace("gene", f"gene_{norm_factor}"), "rb") as f:
        cell_gene_reg_stds_dict = pickle.load(f)
    with open(neighborhood_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"), "rb") as f:
        neighborhood_gene_means_dict = pickle.load(f)
    with open(neighborhood_gene_reg_stds_dictionary_file.replace("gene", f"gene_{norm_factor}"), "rb") as f:
        neighborhood_gene_reg_stds_dict = pickle.load(f)
    
    # Retrieve cell and neighborhood gene means and regularized standard deviations
    cell_gene_means = np.array([cell_gene_means_dict[gene_id] for gene_id in adata.var["ensembl_id"]])
    cell_gene_reg_stds = np.array(
        [cell_gene_reg_stds_dict[gene_id] for gene_id in adata.var["ensembl_id"]]
        )
    neighborhood_gene_means = np.array(
        [neighborhood_gene_means_dict[gene_id] for gene_id in adata.var["ensembl_id"]]
        )
    neighborhood_gene_reg_stds = np.array(
        [neighborhood_gene_reg_stds_dict[gene_id] for gene_id in adata.var["ensembl_id"]]
        )

    # Normalize cell and neighborhood counts
    norm_counts = adata.X - cell_gene_means / cell_gene_reg_stds
    norm_counts_neighborhood = (
        adata.layers["X_neighborhood"] - neighborhood_gene_means / neighborhood_gene_reg_stds
        )
elif norm_method == "shifted_log":
    sc.pp.log1p(adata)
    sc.pp.log1p(adata, layer="X_neighborhood")

    # Retrieve cell and neighborhood gene logmeans
    cell_gene_logmeans = np.array(
        [cell_gene_logmeans_dict[gene_id] for gene_id in adata.var["ensembl_id"]])
    neighborhood_gene_logmeans = np.array(
        [neighborhood_gene_logmeans_dict[gene_id] for gene_id in adata.var["ensembl_id"]]
        )
    
    # Normalize cell and neighborhood counts
    adata.X = adata.X / cell_gene_logmeans
    adata.layers["X_neighborhood"] = adata.layers["X_neighborhood"] / neighborhood_gene_logmeans

In [ ]:
# Keep only protein-coding and miRNA genes
vocab = list(token_dict.keys())
coding_miRNA_ids = [
    key.split("_")[0] for key in list(vocab) if "_cell" in key]
coding_miRNA_dict = dict(zip(coding_miRNA_ids, [True] * len(vocab)))
coding_miRNA_idx = np.where(
    [coding_miRNA_dict.get(gene_id, False) for gene_id in adata.var["ensembl_id"]])[0]
coding_miRNA_adata = adata[:, coding_miRNA_idx]

In [ ]:
cell_idx = 1

In [ ]:
cell_gene_idx = np.argmax(coding_miRNA_adata.X[cell_idx])
coding_miRNA_adata.var_names[cell_gene_idx]
#coding_miRNA_adata.X[0, :].toarray()
#coding_miRNA_adata.X[:, cell_gene_idx].toarray()
#np.mean(coding_miRNA_adata.X[:, cell_gene_idx].toarray())

In [ ]:
#sc.pp.log1p(coding_miRNA_adata)
sq.pl.spatial_scatter(coding_miRNA_adata,
                      color=coding_miRNA_adata.var_names[cell_gene_idx],
                      shape=None) 

In [ ]:
neighborhood_gene_idx = np.argmax(coding_miRNA_adata.layers["X_neighborhood"][cell_idx])
coding_miRNA_adata.var_names[neighborhood_gene_idx]
#coding_miRNA_adata.layers["X_neighborhood"][0, :]
#coding_miRNA_adata.layers["X_neighborhood"][:, neighborhood_gene_idx]
#np.mean(coding_miRNA_adata.layers["X_neighborhood"][:, neighborhood_gene_idx])

In [ ]:
#sc.pp.log1p(coding_miRNA_adata, layer="X_neighborhood")
sq.pl.spatial_scatter(coding_miRNA_adata,
                      layer="X_neighborhood",
                      color=coding_miRNA_adata.var_names[neighborhood_gene_idx],
                      shape=None) 

In [ ]:
# Load tokenized dataset
tokenized_data_folder_path = f"{st_data_gold_folder_path}/{dataset_name}/{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}.dataset"
dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)

In [ ]:
# Get cell and neighborhood gene token rankings
gene_tokens_size = int((model_input_size / 2) - 1)
tokens = np.array(dataset["input_ids"])
coding_miRNA_adata.obsm["cell_gene_token_ranking"] = tokens[:, 1:gene_tokens_size+1]
coding_miRNA_adata.obsm["neighborhood_gene_token_ranking"] = tokens[:, gene_tokens_size+2:]

In [ ]:
# Validate
print(coding_miRNA_adata.var["ensembl_id"][cell_gene_idx])
cell_gene_token_idx = coding_miRNA_adata.obsm["cell_gene_token_ranking"][cell_idx][0]
print(token_dict_reverse[cell_gene_token_idx])

print(coding_miRNA_adata.var["ensembl_id"][neighborhood_gene_idx])
neighborhood_gene_token_idx = coding_miRNA_adata.obsm["neighborhood_gene_token_ranking"][cell_idx][0]
print(token_dict_reverse[neighborhood_gene_token_idx])

### 2.4 Evaluate Tokenizer

In [ ]:
# Load tokenized dataset
norm_factor = "read_depth"
norm_method = "shifted_log" # "seurat_v3", "shifted_log"
model_input_size = 2048
tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}.dataset"
dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)

In [ ]:
dataset

In [ ]:
tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}.dataset"
dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)

#### 2.5.1 Cell Classification

In [7]:
# Load vocab
with open(nichejepa_token_dictionary_file, "rb") as file:
    token_dict = pickle.load(file)

token_dict_reverse = {value: key for key, value in token_dict.items()}

In [8]:
# Get scgpt gene embeddings for NicheJEPA input token ids
def get_classification_tokens(example, top_n):
    example[f"top_{top_n}_cell_tokens"] = [
        token_dict_reverse[token_id].replace("_cell", "")
        for token_id in example["input_ids"][1:top_n+1]]
    example[f"top_{top_n}_neighborhood_tokens"] = [
        token_dict_reverse[token_id].replace("_neighborhood", "")
        for token_id in example["input_ids"][model_input_size+1:top_n+model_input_size+1]]
    #example["top_{top_n}_cell_gene_embeddings_scgpt"] = [
    #    emsemblid_embedding_mapping.get(token, np.zeros(512, dtype=np.float32))
    #    for token in example["top_10_cell_tokens"]]                     
    return example

##### 2.5.1.1 scGPT

Task: cell type classification using scGPT embedded top ranked genes.

In [9]:
### Get scGPT gene embeddings for each gene Ensembl id ###

# Set config
pad_token = "<pad>"
special_tokens = [pad_token, "<cls>", "<eoc>"]
pad_value = -2
n_input_bins = 51

# Get vocab
scgpt_vocab = GeneVocab.from_file(scgpt_vocab_file)
for s in special_tokens:
    if s not in scgpt_vocab:
        scgpt_vocab.append_token(s)

# Retrieve model parameters from config files
with open(scgpt_model_config_file, "r") as f:
    model_configs = json.load(f)
print(f"Resume model from {scgpt_model_file}, the model args will override the "
      f"config {scgpt_model_config_file}.")
embsize = model_configs["embsize"]
nhead = model_configs["nheads"]
d_hid = model_configs["d_hid"]
nlayers = model_configs["nlayers"]
n_layers_cls = model_configs["n_layers_cls"]

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ntokens = len(scgpt_vocab)  # size of vocabulary
model = TransformerModel(ntokens,
                         embsize,
                         nhead,
                         d_hid,
                         nlayers,
                         vocab=scgpt_vocab,
                         pad_value=pad_value,
                         n_input_bins=n_input_bins,)

try:
    model.load_state_dict(torch.load(scgpt_model_file))
    print(f"Loading all model params from {scgpt_model_file}")
except:
    model_dict = model.state_dict()
    pretrained_dict = torch.load(scgpt_model_file)
    pretrained_dict = {k: v
                       for k, v in pretrained_dict.items()
                       if k in model_dict and v.shape == model_dict[k].shape}
    for k, v in pretrained_dict.items():
        print(f"Loading params {k} with shape {v.shape}")
        model_dict.update(pretrained_dict)
        model.load_state_dict(model_dict)
model.to(device)

# Create mapping from Ensembl id to scGPT token id
gene2idx = scgpt_vocab.get_stoi()

metadata_df = pd.read_csv(scgpt_metadata_file)

ensemblid_mapping = dict(zip(metadata_df["feature_name"], metadata_df["feature_id"]))
ensemblid2idx = {ensemblid_mapping[k]: v for k, v in gene2idx.items() if k in ensemblid_mapping}

# Retrieve the data-independent gene embeddings from scGPT
gene_ids = np.array([id for id in ensemblid2idx.values()])
gene_embeddings = model.encoder(torch.tensor(gene_ids, dtype=torch.long).to(device))
gene_embeddings = gene_embeddings.detach().cpu().numpy()

# Store gene embeddings with Ensembl id
emsemblid_embedding_mapping = {}
for i, (key, val) in enumerate(ensemblid2idx.items()):
    emsemblid_embedding_mapping[key] = gene_embeddings[i, :]

Resume model from ../scgpt/pretrained_model/best_model.pt, the model args will override the config ../scgpt/pretrained_model/args.json.
Loading params encoder.embedding.weight with shape torch.Size([60697, 512])
Loading params encoder.enc_norm.weight with shape torch.Size([512])
Loading params encoder.enc_norm.bias with shape torch.Size([512])
Loading params value_encoder.linear1.weight with shape torch.Size([512, 1])
Loading params value_encoder.linear1.bias with shape torch.Size([512])
Loading params value_encoder.linear2.weight with shape torch.Size([512, 512])
Loading params value_encoder.linear2.bias with shape torch.Size([512])
Loading params value_encoder.norm.weight with shape torch.Size([512])
Loading params value_encoder.norm.bias with shape torch.Size([512])
Loading params transformer_encoder.layers.0.self_attn.out_proj.weight with shape torch.Size([512, 512])
Loading params transformer_encoder.layers.0.self_attn.out_proj.bias with shape torch.Size([512])
Loading params tran

In [ ]:
top_n = 10
model_input_size = 2048
scores = []

for norm_method in ["analytic_pearson_residuals",
                    "mean",
                    "nzmedian",
                    "seurat_v3",
                    "shifted_log_mean",
                    "shifted_log",
                   ]:
    if norm_method == "analytic_pearson_residuals":
        normalization = f"{norm_method}"
    if norm_method != "analytic_pearson_residuals":
        norm_factor = "read_depth" # "read_depth", "cell_area"
        normalization = f"{norm_factor}_{norm_method}"
    
    output_file_prefix = f"{dataset_name}_{normalization}_{model_input_size}"
    tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{output_file_prefix}.dataset"
    dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)
    dataset = dataset.map(get_classification_tokens,
                          fn_kwargs={"top_n": top_n},
                          #cache_file_name=dataset_cache_file,
                         )

    X_cell = np.array(
        [[emsemblid_embedding_mapping[token] for token in sequence] for
         sequence in dataset[f"top_{top_n}_cell_tokens"]])
    X_cell = X_cell.reshape(X_cell.shape[0], -1)

    X_neighborhood = np.array(
        [[emsemblid_embedding_mapping[token] for token in sequence] for
         sequence in dataset[f"top_{top_n}_neighborhood_tokens"]])
    X_neighborhood = X_neighborhood.reshape(X_neighborhood.shape[0], -1)

    for token_type in ["cell", "neighborhood"]:
        for label_cat in label_cats:
            label_encoder = LabelEncoder()
            labels = np.array(
                label_encoder.fit_transform(dataset[label_cat]))

            for random_state in range(10):

                np.random.seed(random_state)
            
                indices = np.arange(X_cell.shape[0])
                np.random.shuffle(indices)
                num_train = int(0.8 * len(indices))
                train_indices = indices[:num_train]
                test_indices = indices[num_train:]
                labels_train = labels[train_indices]
                labels_test = labels[test_indices]
                if token_type == "cell":
                    X = X_cell
                else:
                    X = X_neighborhood
                X_train = X[train_indices]
                X_test = X[test_indices]
                
                lr_clf = LogisticRegression(max_iter=1000)
                lr_clf.fit(X_train, labels_train)
                train_score = lr_clf.score(X_train, labels_train)
                test_score = lr_clf.score(X_test, labels_test)
                scores.append({"tokenizer": tokenizer,
                               "normalization": normalization,
                               "token_type": token_type,
                               "clf_task": label_cat,
                               "random_state": random_state,
                               "clf_category": "train",
                               "score": train_score})
                scores.append({"tokenizer": tokenizer,
                               "normalization": normalization,
                               "token_type": token_type,
                               "clf_task": label_cat,
                               "random_state": random_state,
                               "clf_category": "test",
                               "score": test_score})
    
                scores_df = pd.DataFrame(scores)
                scores_df.to_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_scgpt.csv", index=False)

Map:   0%|          | 0/188686 [00:00<?, ? examples/s]

In [ ]:
scores_df = pd.read_csv(f"{results_folder_path}/{dataset_name}/clf_scores.csv")

In [ ]:
# Plot classification accuracies
plt.figure(figsize=(10, 8))
bar_plot = sns.barplot(
    data=scores_df, 
    y='tokenizer',
    x='score',
    hue=scores_df['token_type'] + ' - ' + scores_df['clf_task'], 
    ci=None)
plt.ylabel('Tokenizer') 
plt.xlabel('Score')
plt.title('Test Scores per Tokenizer, Token Type, and Classification Task')
plt.legend(title='Token Type - Classification Task', loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1)
plt.tight_layout()
plt.show()

#### 2.5.2 Neighborhood Ranking

In [ ]:
# Log transform counts as per https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_vizgen_mouse_liver.html#autocorrelation-moran-s-i-score
sc.pp.log1p(adata)

In [ ]:
# Compute spatial autocorrelation score
sq.gr.spatial_autocorr(adata, mode="moran")
if "I" in adata.var.columns:
    del(adata.var["I"])
adata.var = adata.var.join(adata.uns["moranI"][["I"]], how="left")

In [ ]:
# Get most and least autocorrelated genes
num_view = 12
top_autocorr = (
    adata.uns["moranI"]["I"].sort_values(ascending=False).head(num_view).index.tolist())
bot_autocorr = (
    adata.uns["moranI"]["I"].sort_values(ascending=True).head(num_view).index.tolist())

In [ ]:
# Plot most autocorrelated genes
sq.pl.spatial_scatter(
    adata, color=top_autocorr, size=20, cmap="Reds", figsize=(5, 5), shape=None)

In [ ]:
# Plot least autocorrelated genes
sq.pl.spatial_scatter(
    adata, color=bot_autocorr, size=20, cmap="Reds", img=False, figsize=(5, 5), shape=None)

In [ ]:
rank_thresh = 1024

# Sum up ranks for each cell gene token
cell_gene_rank_sums = {}

for token in token_dict.keys():
    if "cell" in token and "ENS" in token:
        cell_gene_rank_sums[token.replace("_cell", "")] = 0

for i, row in enumerate(adata.obsm["cell_gene_token_ranking"]):
    for rank, token_id in enumerate(row[:rank_thresh]):
        if token_id != 0:
            cell_gene_rank_sums[
            token_dict_reverse[token_id].replace("_cell", "")] += (rank_thresh - rank)

cell_gene_rank_file = f"{results_folder_path}/{dataset_name}/cell_gene_rank_sums_{norm_factor}_{norm_method}.pkl"

with open(cell_gene_rank_file, "wb") as file:
    pickle.dump(cell_gene_rank_sums, file)

# Sum up ranks for each neighborhood gene token
neighborhood_gene_rank_sums = {}

for token in token_dict.keys():
    if "neighborhood" in token and "ENS" in token:
        neighborhood_gene_rank_sums[token.replace("_neighborhood", "")] = 0

for i, row in enumerate(adata.obsm["neighborhood_gene_token_ranking"]):
    for rank, token_id in enumerate(row[:rank_thresh]):
        if token_id != 0:
            neighborhood_gene_rank_sums[
            token_dict_reverse[token_id].replace("_neighborhood", "")] += (rank_thresh - rank)

neighborhood_gene_rank_file = f"{results_folder_path}/{dataset_name}/neighborhood_gene_rank_sums_{norm_factor}_{norm_method}.pkl"

with open(neighborhood_gene_rank_file, "wb") as file:
    pickle.dump(neighborhood_gene_rank_sums, file)

adata.var["neigborhood_gene_rank_sums"] = adata.var["ensembl_id"].map(neighborhood_gene_rank_sums).fillna(0)
adata.var["cell_gene_rank_sums"] = adata.var["ensembl_id"].map(cell_gene_rank_sums).fillna(0)

correlation, p_value = spearmanr(adata.var["neigborhood_gene_rank_sums"],
                                 adata.var["I"])
print("Spearman Correlation Coefficient Neighborhood:", correlation)
print("P-value:", p_value)

correlation, p_value = spearmanr(adata.var["cell_gene_rank_sums"],
                                 adata.var["I"])
print("Spearman Correlation Coefficient Cell:", correlation)
print("P-value:", p_value)

### 2.6 Miscellaneous

In [ ]:
print(np.var([2, 4, 8]))
print(np.var([20, 40, 80]))
print(np.var([2, 4, 8] - np.array([np.mean([2, 20]), np.mean([4, 40]), np.mean([8, 80])])))
print(np.var([20, 40, 80] - np.array([np.mean([2, 20]), np.mean([4, 40]), np.mean([8, 80])])))

In [ ]:
print(np.var(np.log([2, 4, 8])))
print(np.var(np.log([20, 40, 80])))
print(np.var(np.log([2, 4, 8]) - np.array([np.mean(np.log([2, 20])), np.mean(np.log([4, 40])), np.mean(np.log([8, 80]))])))
print(np.var(np.log([20, 40, 80]) - np.array([np.mean(np.log([2, 20])), np.mean(np.log([4, 40])), np.mean(np.log([8, 80]))])))

In [ ]:
print(np.var(np.log([2, 4, 8])))
print(np.var(np.log([20, 40, 80])))
print(np.var(np.log([2, 4, 8]) / np.array([np.mean(np.log([2, 20])), np.mean(np.log([4, 40])), np.mean(np.log([8, 80]))])))
print(np.var(np.log([20, 40, 80]) / np.array([np.mean(np.log([2, 20])), np.mean(np.log([4, 40])), np.mean(np.log([8, 80]))])))